# 03 — Серия экспериментов, отчёты и возобновление

Сравниваются два оконных сигнала при одном аллокаторе: max и p95. Три независимых synthetic seeds, три цены. Каждая точка — отдельный сохранённый run. Повторный запуск пропускает завершённые run с совпадающей конфигурацией/кодом/входом и проверенными checksum.

Это стартовый pilot, не доказательство превосходства прогноза: прогнозных моделей в v0.1 нет. Между window-max и p95 возможен компромисс резерва и нарушений.


In [ ]:
from pathlib import Path
from IPython.display import display
from vmforecast import __version__
from vmforecast.settings import Settings

SETTINGS = Settings.load()
ROOT = SETTINGS.root
LAB_DATA_DIR = SETTINGS.data
LAB_RUNS_DIR = SETTINGS.runs
print('vmforecast-lab', __version__)


In [ ]:
import yaml
from vmforecast import make_synthetic, WindowSignal, SimulationConfig
from vmforecast.experiments import run_grid
from vmforecast.reporting import grid_report, paired_difference
CFG = yaml.safe_load((ROOT/'configs'/'window_pilot.yaml').read_text())
synth = dict(CFG['synthetic'])
seeds = synth.pop('seeds')
workloads = [(f'seed_{seed}',make_synthetic(seed=seed,**synth)) for seed in seeds]
signals = {name:WindowSignal(**params) for name,params in CFG['signals'].items()}
config = SimulationConfig(**CFG['simulation'])
OUTPUT_DIR = LAB_RUNS_DIR/'window_pilot'
frame = run_grid(workloads,signals,CFG['prices'],config,OUTPUT_DIR)
display(frame[['case','method','price','migrations_completed','violation_rate','risk_sum']])


In [ ]:
report = grid_report(frame,OUTPUT_DIR)
ci_moves = paired_difference(frame,'max_12','p95_12','migrations_completed')
ci_risk = paired_difference(frame,'max_12','p95_12','violation_rate')
ci_moves.to_csv(OUTPUT_DIR/'paired_migrations.csv',index=False)
ci_risk.to_csv(OUTPUT_DIR/'paired_violation_rate.csv',index=False)
display(ci_moves)
display(ci_risk)
print('Self-contained HTML:',report)


In [ ]:
from IPython.display import Image,display
display(Image(filename=str(OUTPUT_DIR/'migrations_completed.png')))
display(Image(filename=str(OUTPUT_DIR/'violation_rate.png')))


## Что менять дальше

- Цена, размер окна, квантиль, safety_margin и запас ёмкости — через конфигурацию.
- Длительность и overhead миграции — отдельная серия при том же workload.
- Сценарии: seasonal, trend, jump, returning_burst. Непредсказуемый скачок не гарантирует победу одного метода.
- Для trace: замените workloads на сохранённые наборы из notebook 01. Один срез не даёт независимых повторов; для CI нужны разные независимые seeds/блоки/cells, а не повторение одного файла.
- Для будущей карты c×ρ группируйте независимые seed-реплики отдельно для каждого ρ; не усредняйте разные плотности в один CI.

95% bootstrap CI здесь иллюстративен: три seeds мало для окончательного научного вывода. При одном pair CI не рисуется. Для сравнения при равном риске нужны несколько уровней резерва и Pareto analysis; одинаковая цена не означает одинаковый риск.
